In [ ]:
# ---------------------------------------------------------------------------
# Path setup — fixes imports and data paths when notebook runs from subdir
# ---------------------------------------------------------------------------
from pathlib import Path
import sys, os

REPO_ROOT = Path().resolve().parent.parent  # notebooks/analysis/ -> root
os.chdir(REPO_ROOT)                          # all relative paths (cache/, data/, results/, splits/) resolve from root
sys.path.insert(0, str(REPO_ROOT / "src"))   # find rise, crise, synth_gen modules

# CRISE-ID — Checkpoint 3 Demo
### Contrastive RISE for 1:N Face Identification

**10-minute demo script — follow cells in order.**

| Rubric Item | Cell(s) | Time |
|---|---|---|
| Introduction + model definition | 1–2 | ~2 min |
| Forward function | 3 | ~1 min |
| Computation logs + checkpoints | 4 | ~1 min |
| Live inference on a batch | 5 | ~2 min |
| Testing results / screenshots | 6 | ~1 min |
| Comparison to baselines | 7–8 | ~3 min |


---
## 1. Introduction — What is CRISE-ID?

**Problem:** Face recognition systems produce a confidence score, but give no explanation for *why*
they matched a face. For forensic or legal contexts, this is insufficient — you need to know
*which facial regions* drove the decision.

**Prior work (RISE):** Weights each random mask by cosine similarity to the true identity alone.
This is a **self-consistency** check, not a faithfulness test — it ignores competition from
all other gallery identities.

**CRISE-ID:** Replaces the cosine weight with a **softmax probability over all 1,680 gallery identities**.
A mask only gets high weight when the occluded face confidently identifies as the correct person
*relative to every alternative in the gallery* — reflecting the actual 1:N decision boundary.

```
RISE:   w = cos(f(x ∘ m), gallery[true_id])
CRISE:  w = softmax(cos(f(x ∘ m), G) / τ)[true_id]
```

**Dataset:** LFW-deepfunneled — 1,680 gallery identities, 7,484 probe images.
**Backbone:** ArcFace via InsightFace `buffalo_l` (512-dim L2-normalized embeddings, no fine-tuning).


In [ ]:
# ---------------------------------------------------------------------------
# [Rubric: Model definition / Forward function]
# Show crise.py — the core algorithm change
# ---------------------------------------------------------------------------

with open("src/crise.py") as f:
    src = f.read()

# Print the two key sections
sections = src.split("# ------")
for s in sections:
    if "softmax_weight" in s or "TAU" in s[:200]:
        print("=" * 65)
        print(s[:1200])
        print()


In [ ]:
# ---------------------------------------------------------------------------
# [Rubric: Forward function]
# Show the RISE mask loop in rise.py — the computational core
# ---------------------------------------------------------------------------

import inspect
from rise import run_rise

# Print the mask accumulation loop (lines 80-130 approximately)
src_lines = inspect.getsource(run_rise).split("\n")
print("\n".join(src_lines[:80]))


In [ ]:
# ---------------------------------------------------------------------------
# [Rubric: Saved training loss/logs with model checkpoints]
#
# CRISE-ID is not gradient-based, but the computation has exact analogues:
#   "checkpoint" = *_state.json  (stores mask index i; enables resume)
#   "training log" = summary CSV (per-probe margin + saliency paths)
#   "loss curve" = insertion/deletion AUC as a function of N masks
# ---------------------------------------------------------------------------

import os, json, glob
import pandas as pd

# 1. Show a checkpoint file
state_files = glob.glob("results/crise_maps/*_state.json")
print(f"Checkpoint files: {len(state_files)}")
sample_state = json.load(open(state_files[0]))
print(f"Sample checkpoint ({os.path.basename(state_files[0])}):")
print(f"  masks completed : {sample_state['i']} / 1000")
print(f"  failures        : {sample_state['failures']}")

print()

# 2. Show summary log (training log analogue)
log_df = pd.read_csv("results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv")
print(f"Computation log (summary CSV): {len(log_df)} probes")
print(log_df[["true_id", "w_clean", "w_black", "saliency_path"]].head(5).to_string(index=False))

print()

# 3. Saliency maps on disk
sal_maps = glob.glob("results/crise_maps/*_saliency_norm.npy")
print(f"Cached saliency maps (checkpoints): {len(sal_maps)} .npy files")
size_gb = sum(os.path.getsize(f) for f in sal_maps) / 1e9
print(f"Total cache size: {size_gb:.2f} GB")


In [ ]:
# ---------------------------------------------------------------------------
# [Rubric: Run a demo for a batch of data]
# Run CRISE live on 3 probes from the gallery, produce saliency maps.
# ---------------------------------------------------------------------------

import numpy as np
import cv2
import matplotlib.pyplot as plt
from insightface.app import FaceAnalysis
from rise import build_aligned_chip_112, get_embedding_from_chip
from crise import run_crise, TAU

# Setup
app = FaceAnalysis(name="buffalo_l",
                   providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]

G           = np.load("cache/G.npy").astype(np.float32)
gallery_ids = np.load("cache/gallery_ids.npy", allow_pickle=True).tolist()
id_to_index = {n: i for i, n in enumerate(gallery_ids)}

# Pick 3 identities for the live demo
DEMO_IDS = ["George_W_Bush", "Colin_Powell", "Arnold_Schwarzenegger"]
import json as _json
with open("splits/lfw_1N_split.json") as f:
    split = _json.load(f)

probe_paths = {}
for pid in DEMO_IDS:
    probes = split["probes"].get(pid, [])
    if probes:
        probe_paths[pid] = probes[0]

print("Running CRISE on 3 probes...")
OUT_DIR = "results/demo_batch"
os.makedirs(OUT_DIR, exist_ok=True)

results = {}
for i, (pid, ppath) in enumerate(probe_paths.items()):
    print(f"  [{i+1}/3] {pid}")
    out = run_crise(
        true_id=pid, probe_path=ppath,
        G=G, id_to_index=id_to_index, rec=rec, app=app,
        run_seed=99990 + i, out_dir=OUT_DIR,
        tau=TAU, N=200, s=8, p1=0.5,   # N=200 for speed in demo
    )
    results[pid] = out
    rank1 = "RANK-1 ✓" if out["w_clean"] > 0 else "no match"
    print(f"    margin={out['w_clean']:.3f}  {rank1}")

# Plot chips + saliency maps
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for j, (pid, out) in enumerate(results.items()):
    chip_rgb = cv2.cvtColor(out["chip_bgr"], cv2.COLOR_BGR2RGB)
    axes[0, j].imshow(chip_rgb)
    axes[0, j].set_title(pid.replace("_", " "), fontsize=9)
    axes[0, j].axis("off")
    axes[1, j].imshow(out["sal_norm"], cmap="inferno", vmin=0, vmax=1)
    axes[1, j].set_title(f"CRISE saliency\nmargin={out['w_clean']:.3f}", fontsize=8)
    axes[1, j].axis("off")

axes[0, 0].set_ylabel("Aligned chip", fontsize=9, rotation=90)
axes[1, 0].set_ylabel("CRISE map", fontsize=9, rotation=90)
plt.suptitle("Live CRISE-ID inference — 3 probes (N=200 masks for demo speed)", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/demo_batch_3probes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Done.")


In [ ]:
# ---------------------------------------------------------------------------
# [Rubric: Testing log / screenshots]
# Show pre-computed results from the full 1,693-probe CRISE run.
# ---------------------------------------------------------------------------

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

panels = [
    ("results/demo_forensics/Liam/panel_A_saliency_overview.png",
     "Panel A — Saliency overview"),
    ("results/demo_forensics/Liam/panel_B_region_importance.png",
     "Panel B — Per-region importance"),
    ("results/demo_forensics/Liam/panel_C_deletion.png",
     "Panel C — Deletion experiment"),
]

for ax, (path, title) in zip(axes, panels):
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, "Run demo_personal.ipynb\nto generate",
                ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("CRISE-ID Personal Forensics Report — pre-computed outputs", fontsize=12)
plt.tight_layout()
plt.show()

# Also show full eval stats
print("\n=== Full evaluation (1,693 probes, N=1000 masks) ===")
import pandas as pd
log = pd.read_csv("results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv")
eval_df = pd.read_csv("results/crise_maps/eval_margin_auc/eval_margin_auc_steps50_black.csv")
print(f"  Probes evaluated    : {len(eval_df)}")
print(f"  Mean deletion AUC   : {eval_df['del_auc_margin'].mean():.4f}")
print(f"  Mean insertion AUC  : {eval_df['ins_auc_margin'].mean():.4f}")
print(f"  Mean margin (clean) : {eval_df['margin_clean'].mean():.4f}")


---
## Deepfake Forensics — The Core Application

CRISE-ID answers the question confidence scores **cannot**: when a deepfake fools ArcFace,
is it because it replicated genuine identity features, or something else?

**Four-case taxonomy** (two binary axes: fooled ArcFace? × saliency similar to real probe?):

| Case | Fooled? | Saliency similar? | Meaning |
|---|---|---|---|
| A | ✓ | ✓ | Fooled for right reasons — genuine identity replicated |
| **B** | **✓** | **✗** | **Fooled for wrong reasons — non-identity features exploited** |
| C | ✗ | ✓ | Correct features, insufficient transfer strength |
| D | ✗ | ✗ | Complete failure |

**Case B is the forensic finding:** ArcFace confirms identity while using *different* facial
evidence than it uses for the real person. Invisible from confidence scores alone.

**SIM_THRESHOLD = 0.853** (5th percentile of real intra-identity CRISE cosine similarity
across 3,213 genuine same-person probe pairs — principled, not arbitrary).

In [ ]:
# ---------------------------------------------------------------------------
# Deepfake forensics overview — pre-computed results across 876 synthetic probes
# ---------------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os

SIM_THRESHOLD = 0.853   # 5th pct of real intra-identity CRISE sim

meta = pd.read_csv("data/synthetic_probes/metadata.csv")

# Reclassify with principled threshold
def classify(row):
    if pd.isna(row["saliency_cosine_sim"]) or pd.isna(row["rank1_match"]):
        return "?"
    sim_ok = row["saliency_cosine_sim"] >= SIM_THRESHOLD
    fooled  = bool(row["rank1_match"])
    if   fooled and sim_ok:      return "A"
    elif fooled and not sim_ok:  return "B"
    elif not fooled and sim_ok:  return "C"
    else:                         return "D"

meta["case_label"] = meta.apply(classify, axis=1)

# Primary alpha summary
primary = meta[
    ((meta["generation_method"] == "morphing")         & (meta["blend_alpha"] == 0.5)) |
    ((meta["generation_method"] == "sd_img2img")       & (meta["blend_alpha"] == 0.5)) |
    (meta["generation_method"] == "insightface_swap")
].copy()

print("=== Deepfake Forensics: Case Distribution (primary alpha=0.5) ===")
print(f"{'Method':<20}  {'n':>4}  {'Rank-1':>7}  {'Case A':>7}  {'Case B':>7}  {'Case C':>7}  {'Case D':>7}")
print("-" * 70)
for method, grp in primary.groupby("generation_method"):
    n = len(grp)
    r1 = grp["rank1_match"].mean()
    for case in ["A","B","C","D"]:
        pct = (grp["case_label"]==case).mean()
    dist = {c: f"{(grp['case_label']==c).mean()*100:.1f}%" for c in "ABCD"}
    print(f"{method:<20}  {n:>4}  {r1*100:>6.1f}%  "
          f"{dist['A']:>7}  {dist['B']:>7}  {dist['C']:>7}  {dist['D']:>7}")

print()
print(f"Key finding: SD img2img has highest Case B rate (6.5%) at strength=0.5")
print(f"  → AI-generated faces more likely to fool ArcFace for wrong reasons than morphing")
print(f"  → Undetectable from confidence scores — requires CRISE-ID saliency analysis")

# Show pre-computed figures: case distribution + region importance
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (path, title) in zip(axes, [
    ("results/forensics_figures/fig2_cross_method_cases.png",
     "Case distribution by method"),
    ("results/forensics_figures/fig3_region_importance_by_case.png",
     "Per-region importance by case"),
    ("results/forensics_figures/fig_case_B_all_methods.png",
     "Case B: fooled for wrong reasons"),
]):
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, f"Missing:\n{os.path.basename(path)}",
                ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Deepfake Forensics — pre-computed results (876 synthetic probes)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# LIVE: Case A vs Case B side-by-side forensic comparison
#
# Amanda_Beard has both a Case A morphing probe (saliency similar to real)
# and a Case B morphing probe (saliency divergent despite fooling ArcFace).
# ArcFace says "this is Amanda Beard" for BOTH — CRISE-ID reveals the difference.
# ---------------------------------------------------------------------------

import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob, os

SIM_THRESHOLD = 0.853

# ── Load real probe CRISE map ────────────────────────────────────────────
IDENTITY    = "Amanda_Beard"
REAL_PATH   = "data/lfw-deepfunneled/Amanda_Beard/Amanda_Beard_0002.jpg"

real_sal_candidates = glob.glob(
    f"results/crise_maps/crise_tau0p1_{IDENTITY}_{IDENTITY}_0002_*_saliency_norm.npy"
)
assert real_sal_candidates, "Real CRISE map not found"
real_sal  = np.load(real_sal_candidates[0]).astype(np.float32)
real_img  = cv2.imread(REAL_PATH)

# Use pre-computed maps (N=1000) — no GPU needed for this cell
case_a_path  = "data/synthetic_probes/morphing/Amanda_Beard/Amanda_Beard_morph_0p5_1.jpg"
case_a_sal_p = "results/crise_maps/crise_tau0p1_Amanda_Beard_Amanda_Beard_morph_0p5_1_N1000_s8_p0.5_seed420253_saliency_norm.npy"
case_b_path  = "data/synthetic_probes/morphing/Amanda_Beard/Amanda_Beard_morph_0p5_0.jpg"
case_b_sal_p = "results/crise_maps/crise_tau0p1_Amanda_Beard_Amanda_Beard_morph_0p5_0_N1000_s8_p0.5_seed420294_saliency_norm.npy"

# Fallback: search for them
for attr, pattern in [
    ("case_a_sal_p", f"results/crise_maps/crise_tau0p1_{IDENTITY}_{IDENTITY}_morph_0p5_1_*_saliency_norm.npy"),
    ("case_b_sal_p", f"results/crise_maps/crise_tau0p1_{IDENTITY}_{IDENTITY}_morph_0p5_0_*_saliency_norm.npy"),
]:
    found = glob.glob(pattern)
    if found:
        locals()[attr] = found[0]

case_a_sal = np.load(case_a_sal_p).astype(np.float32)
case_b_sal = np.load(case_b_sal_p).astype(np.float32)
case_a_img = cv2.imread(case_a_path)
case_b_img = cv2.imread(case_b_path)

# Compute cosine similarities to real map
def cos_sim(a, b):
    a = a.ravel().astype(np.float32); b = b.ravel().astype(np.float32)
    return float((a/np.linalg.norm(a)) @ (b/np.linalg.norm(b)))

sim_a = cos_sim(real_sal, case_a_sal)
sim_b = cos_sim(real_sal, case_b_sal)

# Load metadata for ArcFace similarity
import pandas as pd
meta = pd.read_csv("data/synthetic_probes/metadata.csv")
af_a = meta[meta["output_path"].str.contains("morph_0p5_1") & meta["identity"].eq(IDENTITY)]["arcface_similarity"].values
af_b = meta[meta["output_path"].str.contains("morph_0p5_0") & meta["identity"].eq(IDENTITY)]["arcface_similarity"].values
af_a = float(af_a[0]) if len(af_a) else 0.0
af_b = float(af_b[0]) if len(af_b) else 0.0

print(f"{'':25s}  {'Real':>8}  {'Case A':>8}  {'Case B':>8}")
print(f"{'ArcFace similarity':25s}  {'(ref)':>8}  {af_a:>8.3f}  {af_b:>8.3f}")
print(f"{'Saliency cos sim (vs real)':25s}  {'1.000':>8}  {sim_a:>8.3f}  {sim_b:>8.3f}")
print(f"{'Fooled ArcFace?':25s}  {'—':>8}  {'YES ✓':>8}  {'YES ✓':>8}")
print(f"{'Case label':25s}  {'—':>8}  {'A':>8}  {'B':>8}")
print()
print(f"  ArcFace gives a confident match for BOTH deepfakes.")
print(f"  CRISE-ID reveals Case B is relying on DIFFERENT facial evidence.")
print(f"  This distinction is invisible from confidence scores alone.")

# ── Figure: 3-column comparison ─────────────────────────────────────────
from insightface.app import FaceAnalysis
from rise import build_aligned_chip_112

_app = FaceAnalysis(name="buffalo_l",
                    providers=["CUDAExecutionProvider","CPUExecutionProvider"])
_app.prepare(ctx_id=0, det_size=(640,640))

def chip(img):
    try:    return build_aligned_chip_112(img, _app)
    except: return np.zeros((112,112,3), dtype=np.uint8)

real_chip  = chip(real_img)
case_a_chip = chip(case_a_img)
case_b_chip = chip(case_b_img)

CASE_COLOR = {"A": "#2ecc71", "B": "#e74c3c"}

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
row_labels = ["Real probe", f"Case A morph (α=0.5)", f"Case B morph (α=0.5)"]
col_labels = ["Aligned chip", "CRISE saliency", "Overlay"]

for j, label in enumerate(col_labels):
    axes[0, j].set_title(label, fontsize=10, fontweight="bold")

for i, (img_bgr, sal, row_label, case_key, af_sim, sal_sim) in enumerate([
    (real_chip,   real_sal,   row_labels[0], None, None, 1.0),
    (case_a_chip, case_a_sal, row_labels[1], "A",  af_a, sim_a),
    (case_b_chip, case_b_sal, row_labels[2], "B",  af_b, sim_b),
]):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[i, 0].imshow(rgb)
    axes[i, 1].imshow(sal, cmap="inferno", vmin=0, vmax=1)
    axes[i, 2].imshow(rgb)
    axes[i, 2].imshow(sal, cmap="inferno", alpha=0.55, vmin=0, vmax=1)

    ylabel = row_label
    if af_sim is not None:
        ylabel += f" arcface={af_sim:.3f}"
    axes[i, 0].set_ylabel(ylabel, fontsize=8, rotation=0, labelpad=85, va="center")

    if case_key:
        color = CASE_COLOR[case_key]
        axes[i, 1].set_title(
            f"Case {case_key} — sal_cos={sal_sim:.3f}  threshold={SIM_THRESHOLD}",
            fontsize=8, color=color, fontweight="bold"
        )

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(
    f"CRISE-ID Deepfake Forensics — {IDENTITY.replace('_',' ')}"
    "ArcFace confirms identity for BOTH deepfakes — CRISE-ID reveals which one used genuine evidence",
    fontsize=11, y=1.01
)
plt.tight_layout()
demo_fig = "results/checkpoint3_deepfake_demo.png"
plt.savefig(demo_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {demo_fig}")


In [ ]:
# ---------------------------------------------------------------------------
# Personal forensics: Panel E — GenAI audit on YOUR own face
# Shows what happens when SD img2img generates an AI version of your probe
# ---------------------------------------------------------------------------

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

panel_e = "results/demo_forensics/Liam/panel_E_genai_forensics_s0p5.png"
if os.path.exists(panel_e):
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(mpimg.imread(panel_e))
    ax.axis("off")
    ax.set_title(
        "Panel E — GenAI Forensics: AI-generated version of your own face\n"
        "Does ArcFace identify it as you, and is it using the same facial evidence?",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
    print("Panel E: pre-computed from demo_personal.ipynb")
    print("  Each row = one real probe + its SD img2img synthetic version")
    print("  Columns: real chip | real CRISE | synth chip | synth CRISE | Case label")
else:
    print(f"Panel E not found at {panel_e}")
    print("Run demo_personal.ipynb Panel E cells first.")


In [ ]:
# ---------------------------------------------------------------------------
# [Rubric: Complete comparison results to baselines]
# [Rubric: Multiple baselines and validation metrics]
# [Rubric: Meaningful performance improvement]
# ---------------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

comp = pd.read_csv("results/crise_maps/eval_margin_auc/rise_vs_crise_comparison.csv")
print("=" * 60)
print(" RISE vs. CRISE — Quantitative Comparison")
print("=" * 60)
print(comp.to_string(index=False))
print()

rise_del = comp.loc[comp["Method"].str.contains("RISE"), "Deletion AUC"].values[0]
crise_del = comp.loc[comp["Method"].str.contains("CRISE"), "Deletion AUC"].values[0]
rise_ins = comp.loc[comp["Method"].str.contains("RISE"), "Insertion AUC"].values[0]
crise_ins = comp.loc[comp["Method"].str.contains("CRISE"), "Insertion AUC"].values[0]

del_impr = (rise_del - crise_del) / abs(rise_del) * 100
ins_impr = (crise_ins - rise_ins) / abs(rise_ins) * 100

print(f"  Deletion AUC  improvement : {del_impr:+.1f}%  ({rise_del:.4f} → {crise_del:.4f})")
print(f"  Insertion AUC improvement : {ins_impr:+.1f}%  ({rise_ins:.4f} → {crise_ins:.4f})")
print()
print("  Sanity checks:")
print(f"    CRISE deletion < RISE deletion : {crise_del < rise_del}  ✓")
print(f"    CRISE insertion > RISE insertion: {crise_ins > rise_ins}  ✓")
print("=" * 60)


In [ ]:
# ---------------------------------------------------------------------------
# Comparison figure — bar chart + deepfake forensics summary
# ---------------------------------------------------------------------------

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

fig = plt.figure(figsize=(15, 5))

# ── Left: AUC comparison bar chart ──────────────────────────────────────
ax1 = fig.add_subplot(1, 3, 1)
methods = ["RISE\n(baseline)", "CRISE-ID\n(ours, τ=0.1)"]
del_vals = [rise_del, crise_del]
ins_vals = [rise_ins, crise_ins]
x = np.arange(2); w = 0.3

bars_d = ax1.bar(x - w/2, del_vals, w, label="Deletion AUC ↓", color=["#aec7e8", "#4E79A7"], alpha=0.85)
bars_i = ax1.bar(x + w/2, ins_vals, w, label="Insertion AUC ↑", color=["#ffbb78", "#F28E2B"], alpha=0.85)

for bar in [*bars_d, *bars_i]:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax1.set_xticks(x); ax1.set_xticklabels(methods, fontsize=9)
ax1.set_ylabel("AUC (identification margin)"); ax1.set_title("CRISE vs. RISE baseline")
ax1.legend(fontsize=8); ax1.grid(axis="y", alpha=0.3)
ax1.text(0.5, -0.18,
         f"Del: {del_impr:+.1f}%   Ins: {ins_impr:+.1f}%",
         ha="center", transform=ax1.transAxes, fontsize=9,
         color="#2ca02c", fontweight="bold")

# ── Middle: forensics case distribution ─────────────────────────────────
ax2 = fig.add_subplot(1, 3, 2)
try:
    import pandas as pd
    meta = pd.read_csv("data/synthetic_probes/metadata.csv")
    primary = meta[
        ((meta["generation_method"] == "morphing") & (meta["blend_alpha"] == 0.5)) |
        ((meta["generation_method"] == "sd_img2img") & (meta["blend_alpha"] == 0.5)) |
        (meta["generation_method"] == "insightface_swap")
    ]
    case_counts = primary.groupby(["generation_method", "case_label"]).size().unstack(fill_value=0)
    case_pct = case_counts.div(case_counts.sum(axis=1), axis=0) * 100

    x3 = np.arange(len(case_pct))
    colors = {"A": "#2ecc71", "B": "#e74c3c", "C": "#f39c12", "D": "#95a5a6"}
    bottom = np.zeros(len(case_pct))
    for case in ["A", "B", "C", "D"]:
        if case in case_pct.columns:
            ax2.bar(x3, case_pct[case], bottom=bottom,
                    label=f"Case {case}", color=colors[case], alpha=0.85)
            bottom += case_pct[case].values
    ax2.set_xticks(x3)
    ax2.set_xticklabels([m.replace("_", "\n") for m in case_pct.index], fontsize=8)
    ax2.set_ylabel("% of synthetic probes")
    ax2.set_title("Deepfake forensics\n(A/B/C/D case distribution)")
    ax2.legend(fontsize=8, loc="upper right"); ax2.grid(axis="y", alpha=0.3)
except Exception as e:
    ax2.text(0.5, 0.5, str(e), ha="center", va="center", transform=ax2.transAxes, fontsize=8)
    ax2.set_title("Deepfake case distribution")

# ── Right: personal forensics panel A ────────────────────────────────────
ax3 = fig.add_subplot(1, 3, 3)
panel_path = "results/demo_forensics/Liam/panel_A_saliency_overview.png"
if os.path.exists(panel_path):
    ax3.imshow(mpimg.imread(panel_path))
ax3.set_title("Personal audit — which face regions\ndoes ArcFace rely on for you?", fontsize=9)
ax3.axis("off")

plt.suptitle("CRISE-ID: Contrastive RISE for 1:N Face Identification",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/checkpoint3_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/checkpoint3_comparison.png")
